In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import ignite.distributed as idist


# 1. Define a dummy model
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc = nn.Linear(10, 5)

    def forward(self, x):
        # DataParallel automatically splits the input batch across your GPUs
        print(f"Processing batch of shape {x.shape} on device: {x.device}")
        return self.fc(x)


# 2. Check GPU availability and set primary device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 3. Instantiate and move the model to the primary device
model = SimpleModel()
model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

x_data = torch.randn(100, 10)
y_data = torch.randn(100, 5)
dataset = TensorDataset(x_data, y_data)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# 4. Wrap the model with DataParallel if multiple GPUs are available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for parallel training.")
    # By default, this uses all available GPUs
    model = idist.auto_model(model)
    # optimizer = idist.auto_optim(optimizer)
    # print(dataloader)
    # dataloader = idist.auto_dataloader(dataset)
    # print(dataloader)

# 5. Create dummy dataset and data loader
# Ensure your batch size is large enough to be split across all your GPUs
x_data = torch.randn(100, 10)
y_data = torch.randn(100, 5)
dataset = TensorDataset(x_data, y_data)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# 6. Dummy execution loop
criterion = nn.MSELoss()
# optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for inputs, targets in dataloader:
    # Move tensors to the primary device
    inputs, targets = inputs.to(device), targets.to(device)

    # Forward pass
    outputs = model(inputs)
    loss = criterion(outputs, targets)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Execution completed successfully.")

In [ ]:
import os

os.environ["NCCL_P2P_DISABLE"] = "1"

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import ignite.distributed as idist
import ignite


class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc = nn.Linear(10, 5)

    def forward(self, x):
        # DataParallel automatically splits the input batch across your GPUs
        print(f"Processing batch of shape {x.shape} on device: {x.device}")
        return self.fc(x)


def training(rank):
    device = idist.device()

    x_data = torch.randn(10, 10)
    y_data = torch.randn(10, 5)
    dataset = TensorDataset(x_data, y_data)
    # dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

    dataloader = idist.auto_dataloader(dataset, batch_size=10, shuffle=True)

    model = SimpleModel()
    model = idist.auto_model(model)
    print(f"TYPE: {type(model)}")
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    optimizer = idist.auto_optim(optimizer)

    def train_step(engine, batch):
        # print(f"TYPE: {type(model)}")
        data = batch[0].to(device)
        target = batch[1].to(device)
        output = model(data)
        loss_val = criterion(output, target)
        return loss_val

    trainer = ignite.engine.Engine(train_step)
    trainer.run(dataloader, max_epochs=10)


# with idist.Parallel(backend="gloo", nproc_per_node=2) as parallel:
#     parallel.run(training)

training(0)